In [37]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [38]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [43]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses":5,
    #"allowed_domains":["thequantuminsider.com"] # allowed domains focuses the search onto a specfic list of domains
}

In [44]:
messages = []
add_user_message(
    messages,
    """
    What is the latest news on Quantum computing today?
    """,
)
response = chat(messages, tools = [web_search_schema])

In [78]:
for block in response.content:
    if block.type ==  "web_search_tool_result":
        print(block.content[0].source)
        #print(block.title)
        #print(block.citations)

AttributeError: 'WebSearchResultBlock' object has no attribute 'source'

In [49]:
for block in response.content:
    print(block)

ServerToolUseBlock(id='srvtoolu_01FXmWyKiHXcmk3rv5wa4Uh9', caller=None, input={'query': 'quantum computing news today'}, name='web_search', type='server_tool_use')
WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EqIQCioIEhgCIiRjODA0ZGRmNi0wMzYwLTQwZWMtYmM5Zi0zNzg4Y2QxNzEzNDASDM29En4T3y1BgxIJuBoMId1C7x6nE7UVbEccIjB8OtrGntJuY3qup1DXoa+fr8/GVVVmuXtRuuZRB1DTidO3NXQ0sfVDzV5t8gS2rUwqpQ8Q49/sJBWyfj6L3HS3/LdR2sLUqvqHm/ZXHXVzXevxOC/6B3bskVb7J9MV2BAiy+VKl0nfY/08zKYPsaFLmhy6onb6jvpo7t2FPyi+vub0g4M12RQMZqGlA+LarP3xNziMMj2eL7tclbWRVcumzu4cOhNr8C9he9gu6fwBbiBJMeyzPxml3zMuVawXH/5GSn/Czaq5xCcG/HAq82WJtifatf6wLr8O+LzkGcQl8a4LnLgc3/FT8KNtRRPdGQxwkN7uJNfxkVgWOq8rTRJv6ve1QYBuqbx/faUTWIMm0oGC9vtBdkeLSc8lgH5b6Yp5SZ9eBwbrIE3gU+ECMYpyLjngWDL6x7CiEPF23yK8Ghg4tntwj/r30BYp6jQV/yOlfUOn9iqjG9zuuVFlUIR9yM85D8hDHItaI2emXwu1FSTrlnGVL0REC3dhsvmWGNjqS+GjO+C4FTytK+zRtCt9IaGoxa6ATk3u7O6DETdG74IBIpCeBOhWJMuasRxqWF4UDzx7vS00W3EbSxmp+qQoI/D5Dtz3A+qxhYMNdXsAUm5kHU